# Time Analysis

Response-timing / question-duration analysis.

**Reads:** `data/individual/ (one participant, per session)`  
**Shared code:** `import mms` (loaders in `mms.io`, metrics in `mms.hrv` / `mms.stats`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PSY = '../data/individual/psychometric'

# load data
psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

# Convert datetimes
psychometric_01['Question Start Time'] = pd.to_datetime(psychometric_01['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Start Time'] = pd.to_datetime(psychometric_02['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Start Time'] = pd.to_datetime(psychometric_03['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)

psychometric_01['Question Answer Time'] = pd.to_datetime(psychometric_01['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Answer Time'] = pd.to_datetime(psychometric_02['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Answer Time'] = pd.to_datetime(psychometric_03['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)

# answer duration
psychometric_01['answer_duration'] = (psychometric_01['Question Answer Time'] - psychometric_01['Question Start Time']).dt.total_seconds() / 60
psychometric_02['answer_duration'] = (psychometric_02['Question Answer Time'] - psychometric_02['Question Start Time']).dt.total_seconds() / 60
psychometric_03['answer_duration'] = (psychometric_03['Question Answer Time'] - psychometric_03['Question Start Time']).dt.total_seconds() / 60

psychometric_01['Session'] = 'Session 01'
psychometric_02['Session'] = 'Session 02'
psychometric_03['Session'] = 'Session 03'

all_sessions = pd.concat([psychometric_01, psychometric_02, psychometric_03])

session_stats = all_sessions.groupby('Session')['answer_duration'].agg(['count', 'sum', 'std']).reset_index()
session_stats = session_stats.rename(columns={'count': 'Count', 'sum': 'Total Timing (minutes)', 'std': 'Standard Deviation (minutes)'})

# display results
print("Total Timing During 3 Sessions")
print(session_stats)

# Plot answer duration
plt.figure(figsize=(12, 6))
sns.boxplot(data=all_sessions, x='Session', y='answer_duration')
plt.title('Comparison of Answer Duration Across Sessions')
plt.ylabel('Answer Duration (minutes)')
plt.xlabel('Session')
plt.show()
plt.close()

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

PSY = '../data/individual/psychometric'

# load data
psy_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psy_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psy_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

for df in [psy_01, psy_02, psy_03]:
    df['Question Start Time'] = pd.to_datetime(df['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
    df['Question Answer Time'] = pd.to_datetime(df['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
    df['duration_s'] = (df['Question Answer Time'] - df['Question Start Time']).dt.total_seconds()

sessions = {
    'Session 01': psy_01['duration_s'],
    'Session 02': psy_02['duration_s'],
    'Session 03': psy_03['duration_s']
}

# normality tests
print("=== Shapiro-Wilk Tests ===\n")
for name, dur in sessions.items():
    stat, p = stats.shapiro(dur)
    normal = "normal" if p > 0.05 else "non-normal"
    print(f"{name}: W={stat:.4f}, p={p:.4f} ({normal})")

# Kruskal-Wallis
print("\n=== Kruskal-Wallis Test ===")
h, p = stats.kruskal(*sessions.values())
sig = "*" if p < 0.05 else "ns"
print(f"H={h:.3f}, p={p:.4f} {sig}")

# by question type
print("\n=== Duration by Question Type ===\n")
for q_type in psy_01['Type'].unique():
    means = []
    for df in [psy_01, psy_02, psy_03]:
        m = df[df['Type'] == q_type]['duration_s'].mean()
        means.append(m)
    print(f"{q_type}: {[f'{m:.1f}s' for m in means]}")

# violin by type
all_data = pd.concat([
    psy_01.assign(Session='01'),
    psy_02.assign(Session='02'),
    psy_03.assign(Session='03')
])

fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=all_data, x='Type', y='duration_s', hue='Session', ax=ax, inner='box', palette='Set2')
ax.set_ylabel('Duration (seconds)')
ax.set_xlabel('Question Type')
ax.set_title('Answer Duration by Question Type Across Sessions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
plt.close()